# From real sky images to GPU workers: cuPhoton with Dragon

A 35-minute walkthrough for the SC26 cuPhoton and Dragon session. We begin with
observational FITS pixels, add controlled changes, and follow the same data through
loading, registration, subtraction, candidate fitting, and classification.
At each distributed stage, compare an ordinary Python calculation with a visible
`cuphoton` command launched by Dragon. Finish by retaining the complete pipeline
inside each GPU worker and selecting the newer native CUDA C++ implementations.

| Minutes | Question |
|---|---|
| 0–3 | What are the observations, injected signals, and independent work items? |
| 3–8 | How do FITS loading and WCS registration prepare a pair? |
| 8–13 | Can several workers subtract the same batch correctly? |
| 13–19 | What does a candidate fit tell us, and where does it fail? |
| 19–23 | How do we apply a small trained classifier across GPUs? |
| 23–31 | What changes when the stages share one persistent GPU worker? |
| 31–35 | Where do native implementations remove runtime kernel compilation? |

The optional final section adds 5–7 minutes of automatic detection and uncertainty
analysis. Preparation, downloads, native builds, and training happen **before**
the session; follow [README.md](README.md). Run cells in order. All computation
requires Linux, CUDA 13, and the documented Python 3.12 environment.


## 1. Choose a prepared workload and launch mode

The background pixels come from real observations. The moving sources, sharp
artifacts, and labels are controlled injections; this is not a discovery of
astrophysical transients. The prepared model is trained on those injections and
is useful for demonstrating the execution path, not survey-ready classification.
Its held-out results and exact training provenance remain beside the checkpoint.

Set `CUPHOTON_DEMO_ROOT` to the prepared directory and
`CUPHOTON_DEMO_WORKERS` for your allocation. The executable `%%bash` cells
use one host (`dragon -s`) by default. Each distributed stage also has complete
Slurm and SSH commands: replace the single-host command when using a cluster.
Multinode runs require identical absolute paths for the environment, inputs,
checkpoint, and outputs on every node.

Python setup exports paths to the shell. The command cells contain the actual
CLI arguments, followed by Python cells that inspect their results. Each command
has a 10-minute timeout, a regular-file log, and a GNU `time` wall-clock record.
The timeout bounds the launcher; after a failure, inspect its owned workers
before rerunning. The cell shows the command and the last 40 log lines; the complete log remains
under `RUNS`. Linux Bash, GNU `time` (`/usr/bin/time`), and `timeout` are required.

In [ ]:
# SPDX-FileCopyrightText: Copyright (c) 2026 NVIDIA CORPORATION & AFFILIATES. All rights reserved.
#
# SPDX-License-Identifier: Apache-2.0

import gc
import json
import os
from pathlib import Path
import sys
import time
import uuid

import numpy as np
import pandas as pd
import yaml
from IPython.display import display

# A served output copy may live elsewhere; point it at the source checkout.
repo_override = os.environ.get("CUPHOTON_DEMO_REPO")
search_roots = ([Path(repo_override).expanduser().resolve()] if repo_override
                else [Path.cwd(), *Path.cwd().parents])
EXAMPLE = next((
    candidate
    for parent in search_roots
    for candidate in (parent, parent / "examples" / "dragon-imaging-pipeline")
    if (candidate / "demo.py").is_file()
), None)
if EXAMPLE is None:
    raise RuntimeError(
        "Cannot find examples/dragon-imaging-pipeline/demo.py. Start Jupyter from "
        "the cuPhoton checkout, or set CUPHOTON_DEMO_REPO to its absolute path."
    )
REPO = EXAMPLE.parent.parent
sys.path.insert(0, str(EXAMPLE))
from demo import compare_arrays, image_grid, load_json, summarize_run

DEMO = Path(os.environ.get(
    "CUPHOTON_DEMO_ROOT", REPO / "scratch" / "sc26-dragon"
)).expanduser().resolve()
if not (DEMO / "prepared.json").is_file():
    raise FileNotFoundError(
        f"No prepared.json in {DEMO}. Set CUPHOTON_DEMO_ROOT to the completed output "
        "of prepare.py; see README.md."
    )
prepared = load_json(DEMO / "prepared.json")
WORKERS = int(os.environ.get("CUPHOTON_DEMO_WORKERS", "1"))
NODES = int(os.environ.get("CUPHOTON_DEMO_NODES", "2"))
HOSTFILE = str(Path(os.environ.get(
    "CUPHOTON_DEMO_HOSTFILE", DEMO / "dragon-hosts.txt"
)).expanduser().resolve())
assert WORKERS > 0 and NODES > 0
SESSION = f"notebook-{time.strftime('%Y%m%d-%H%M%S')}-{uuid.uuid4().hex[:6]}"
RUNS = DEMO / "runs" / SESSION
RUNS.mkdir(parents=True, exist_ok=False)
# %%bash inherits these environment variables; PATH selects this kernel's tools.
os.environ["PATH"] = str(Path(sys.executable).parent) + os.pathsep + os.environ["PATH"]
os.environ.update({
    "RUNS": str(RUNS), "WORKERS": str(WORKERS), "NODES": str(NODES), "HOSTFILE": HOSTFILE,
    "PAIRS_MANIFEST": prepared["pairs_manifest"],
    "PIPELINE_MANIFEST": prepared["pipeline_manifest"],
    "NATIVE_PIPELINE_MANIFEST": prepared["native_pipeline_manifest"],
    "MODEL_DIR": prepared["model_dir"], "DEMO_DATASET": prepared["dataset_dir"],
    "XFIT_INPUT": prepared["xfit_input"], "XFIT_FEATURES": prepared["xfit_feature_dir"],
})
print({"data": str(DEMO), "runs": str(RUNS), "workers": WORKERS, "python": sys.executable})

### Workload manifests and Dragon placement are separate inputs

The YAML below lists **whole aligned image pairs** for xPois. It does not select
hosts. Dragon selects hosts through `-s`, a Slurm allocation, or a host file;
cuPhoton binds workers to the available GPUs. xFit and xScan divide their inputs
into candidate or minibatch chunks. The resident pipeline uses a JSON manifest
containing a checkpoint, numerical settings, and pairs with candidate positions.

Dragon queues carry control messages and completion metadata. Workers read image
pixels from shared files. They do not send image arrays through those queues.


In [ ]:
pairs_path = Path(prepared["pairs_manifest"])
pairs_payload = yaml.safe_load(pairs_path.read_text())
pairs = pairs_payload["pairs"]
print(yaml.safe_dump({"schema": pairs_payload["schema"], "pairs": pairs[:2]},
                     sort_keys=False))
print(f"{len(pairs)} independent pairs; at most {WORKERS} GPU workers")
pipeline_path = Path(prepared["pipeline_manifest"])
pipeline_payload = load_json(pipeline_path)
science_settings = pipeline_payload["configuration"]
xpois_settings = science_settings["xpois"]
print("Pipeline stages:", "xPois → xFit → xScan")
print("Model:", prepared["model_dir"])
print("Numerical xFit policy:", science_settings["xfit"].get("solver_version", "v0.2"))

# The literal Bash examples below use these published demo settings.
assert xpois_settings["kernel_shape"] == [15, 15]
assert xpois_settings["basis_sigmas"] == [1.5] and xpois_settings["basis_degrees"] == [0]
assert xpois_settings["background_degree"] == 0 and not xpois_settings["flux_conserve"]
assert science_settings["xfit"]["max_evaluations"] == 400
assert prepared["config"]["batch_size"] == 32
assert prepared["config"]["inference_split"] == "test"

### Check the numerical environment

Torch is imported before CuPy initializes CUDA. The notebook owns one local GPU
for the Python comparisons; each Dragon launch starts a fresh coordinator and
binds its workers before their numerical libraries initialize. Explicit backends
make a missing GPU or native extension an error instead of silently changing the
calculation to a CPU run.


In [ ]:
import torch
import cupy as cp
import cuphoton
from astropy.io import fits
from astropy.wcs import WCS

assert torch.cuda.is_available()
torch.cuda.set_device(0)
cp.cuda.Device(0).use()
print({"cuPhoton": cuphoton.__version__, "Torch": torch.__version__,
       "CuPy": cp.__version__, "GPU": torch.cuda.get_device_name(0),
       "CUDA runtime": cp.cuda.runtime.runtimeGetVersion()})
first = pairs[0]
REFERENCE = Path(first["reference"])
TARGET = Path(first["target"])
HDU = int(first.get("reference_hdu", 1))
data_manifest = load_json(prepared["data_manifest"])
source_info = data_manifest["source"]
display(pd.Series({key: source_info[key] for key in ("title", "units", "credit", "dataset_doi")}))
_ = image_grid([fits.getdata(data_manifest["crop"]["path"], 1)],
               ["M51: original archival crop, 3.6 microns"], columns=1)

assert HDU == int(first.get("target_hdu", 1)) == int(first.get("variance_hdu", 1)) == 1
os.environ.update({
    "REFERENCE": str(REFERENCE), "TARGET": str(TARGET),
    "VARIANCE": first["variance"], "FIT_MASK": first["fit_mask"],
})

## 2. Read and inspect real FITS pixels · 3–8 minutes

xDR returns device arrays. The receipt below identifies the selected reader and
HDU; it is not proof of GPUDirect Storage or a hardware decompression engine.
Display downloads are intentional here. The later resident pipeline keeps its
large intermediate arrays on the worker GPU.


In [ ]:
from cuphoton.core.fits_io import inspect_fits_images, read_fits_images

planes = inspect_fits_images(REFERENCE)
display(pd.DataFrame([{"hdu": p.hdu, "shape": p.shape,
                       "dtype": str(p.dtype), "compression": p.compression}
                      for p in planes]))
reference_read = read_fits_images(REFERENCE, [HDU], reader="xdr", device=True)
target_read = read_fits_images(TARGET, [HDU], reader="xdr", device=True)
reference_device = reference_read.arrays[0]
target_device = target_read.arrays[0]
reference = cp.asnumpy(reference_device)
target = cp.asnumpy(target_device)
print(reference_read.metadata())
_ = image_grid([reference, target], ["Observed background + reference injections",
                                "Controlled later scene"],
           columns=2, shared_scale=True)


### Register onto an explicit reference WCS

The preparation step also saves a shifted copy of the first target with a
corresponding WCS. xRep uses the WCS to return it to the reference pixel grid.
The aligned pair used below for subtraction was prepared in advance so every
worker reads the same immutable bytes.

FITS loading and xRep are local operations in this walkthrough. xRep has a normal
CLI, but no `--executor dragon`; we do not manufacture a distributed equivalent.
Once registration is complete, Dragon distributes complete independent pairs.


In [ ]:
from cuphoton.xrep import BBox, Grid, build_stack_spec_from_fits, reproject_stack

reprojection = data_manifest["reprojection"]
RAW_TARGET = Path(reprojection["raw_target"])
target_wcs = WCS(fits.getheader(REFERENCE, HDU), naxis=2)
height, width = reference.shape
sources, stack_spec = build_stack_spec_from_fits(
    [REFERENCE, RAW_TARGET], hdu=HDU,
    grid=Grid.from_wcs(target_wcs), output_bbox=BBox(0, 0, width, height),
    interpolation="lanczos3", mapping_grid_step=1,
    fits_reader="xdr", device=True,
)
aligned = reproject_stack(sources, stack_spec, backend="cupy")
raw_target = fits.getdata(RAW_TARGET, HDU)
_ = image_grid([raw_target, aligned.images[1], target],
           ["Shifted detector pixels", "xRep: reference grid", "Prepared aligned target"],
           shared_scale=True)

os.environ["RAW_TARGET"] = str(RAW_TARGET)
# A local view makes the known three-pixel shift visible at notebook scale.
anchor = data_manifest["pairs"][0]["candidates"][0]
x, y = int(anchor["x"]), int(anchor["y"])
view = np.s_[y - 25:y + 26, x - 25:x + 26]
_ = image_grid([raw_target[view], aligned.images[1][view],
                (raw_target - aligned.images[1])[view]],
               ["Before WCS registration: 51 px", "After WCS registration: 51 px",
                "Before minus after: own scale"], diverging=[False, False, True])

In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xrep-cupy.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xrep-cupy.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  cuphoton xrep reproject-stack \
    --inputs "$REFERENCE,$RAW_TARGET" --hdu 1 \
    --target-wcs "$REFERENCE" --target-hdu 1 \
    --backend cupy --fits-reader xdr --interpolation lanczos3 \
    --mapping-grid-step 1 --write-fits \
    --output-dir "$RUNS" --name xrep-cupy >"$RUNS/xrep-cupy.log" 2>&1

In [ ]:
print("xrep-cupy: launcher-to-exit seconds:", float((RUNS / "xrep-cupy.elapsed").read_text()))
cli_aligned = np.load(RUNS / "xrep-cupy" / "artifacts" / "reprojected_stack.npy")
print(compare_arrays(aligned.images, cli_aligned, label="xRep Python / CLI"))

## 3. Match the PSF and subtract · 8–13 minutes

Fit a kernel and differential background so the reference resembles the target,
then form **target − matched reference**. The injected changing sources are
excluded by the prepared fit mask. The basis widths are inputs, not parameters
optimized by this solve. Both routes use the same basis, mask and variance.


In [ ]:
from cuphoton.xpois import GaussianBasisComponent, solve_constant_kernel_device

components = [GaussianBasisComponent(sigma=s, degree=d)
              for s, d in zip(xpois_settings["basis_sigmas"],
                              xpois_settings["basis_degrees"], strict=True)]
variance = fits.getdata(first["variance"], int(first.get("variance_hdu", 1)))
fit_mask_path = Path(first["fit_mask"])
fit_mask = (np.load(fit_mask_path, allow_pickle=False) if fit_mask_path.suffix == ".npy"
            else fits.getdata(fit_mask_path, 1)).astype(bool)
solve_options = dict(
    kernel_shape=tuple(xpois_settings["kernel_shape"]),
    variance=cp.asarray(variance), fit_mask=cp.asarray(fit_mask),
    background_degree=xpois_settings["background_degree"],
    flux_conserve=xpois_settings["flux_conserve"],
)
subtraction = solve_constant_kernel_device(
    reference_device, target_device, components, backend="cupy", **solve_options
)
difference = cp.asnumpy(subtraction.residual)
print({"backend": subtraction.backend, "fit_pixels": int(subtraction.fit_pixel_count),
       "chi2": float(subtraction.chi2), "kernel_sum": float(subtraction.kernel.sum())})
_ = image_grid([reference, target, cp.asnumpy(subtraction.matched)],
           ["Reference", "Target", "Matched reference"], shared_scale=True)
_ = image_grid([cp.asnumpy(subtraction.kernel), difference, fit_mask],
           ["Fitted kernel", "Difference: target − matched", "Kernel-fit pixels"],
           diverging=[False, True, False])


### Run the same subtraction over the image-pair batch with Dragon

One pair is one item; one GPU worker can process several items. Increasing the
number of candidates within a single pair does not create more xPois workers.
The single-host cell and the cluster alternatives use the same numerical arguments.

In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xpois-dragon.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xpois-dragon.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp cuphoton xpois fit-batch \
    --executor dragon --backend cupy --manifest "$PAIRS_MANIFEST" --fits-reader xdr \
    --kernel-height 15 --kernel-width 15 --basis-sigmas 1.5 --basis-degrees 0 \
    --background-degree 0 --output-dir "$RUNS" --name xpois-dragon \
    --max-workers "$WORKERS" >"$RUNS/xpois-dragon.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp cuphoton xpois fit-batch \
    --executor dragon --backend cupy --manifest "$PAIRS_MANIFEST" --fits-reader xdr \
    --kernel-height 15 --kernel-width 15 --basis-sigmas 1.5 --basis-degrees 0 \
    --background-degree 0 --output-dir "$RUNS" --name xpois-dragon \
    --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp cuphoton xpois fit-batch \
    --executor dragon --backend cupy --manifest "$PAIRS_MANIFEST" --fits-reader xdr \
    --kernel-height 15 --kernel-width 15 --basis-sigmas 1.5 --basis-degrees 0 \
    --background-degree 0 --output-dir "$RUNS" --name xpois-dragon \
    --max-workers "$WORKERS"
```

In [ ]:
print("xpois-dragon: launcher-to-exit seconds:", float((RUNS / "xpois-dragon.elapsed").read_text()))
_ = summarize_run(RUNS / "xpois-dragon")
cli_difference = np.load(RUNS / "xpois-dragon" / "items" / first["id"] /
                         "artifacts" / "residual.npy")
print(compare_arrays(difference, cli_difference, label="xPois Python / Dragon"))
_ = image_grid([difference, cli_difference, difference - cli_difference],
           ["Python difference", "Dragon difference", "Numerical difference"],
           diverging=True)

## 4. Fit candidate stamps · 13–19 minutes

The prepared candidates contain known injections and controls; this section is
not a source detector. Each input stamp, ID and initial condition is frozen in
an NPZ file derived from the same subtraction settings. A Gaussian dipole models
a positive and a negative lobe. A sharp artifact, blank stamp, or single brightening
source can violate that model even when the optimizer returns parameters.

Keep **solver policy v0.2**, float64, and candidate chunk size identical in the
Python and Dragon runs. Inspect convergence and uncertainty validity separately.
The final optional chain adds detection, quality flags, and single-source refits.


In [ ]:
from cuphoton.xfit import GaussianDipoleModel, LMConfig, fit_dipoles
from cuphoton.xfit.io import load_xfit_dataset

xfit_data = load_xfit_dataset(prepared["xfit_input"], model="gaussian", mode="difference")
CHUNK = int(prepared.get("config", {}).get("chunk_size", 32))
MAX_EVALUATIONS = int(science_settings["xfit"]["max_evaluations"])
assert CHUNK == 32 and MAX_EVALUATIONS == 400
fit_config = LMConfig(solver_version="v0.2", max_evaluations=MAX_EVALUATIONS)
python_fits = [
    fit_dipoles(
        xfit_data.images[start:start + CHUNK], model="gaussian", mode="difference",
        initial=None if xfit_data.initial is None else xfit_data.initial[start:start + CHUNK],
        backend="cupy", config=fit_config,
    )
    for start in range(0, xfit_data.batch_size, CHUNK)
]
python_parameters = np.concatenate([result.parameters for result in python_fits])
python_converged = np.concatenate([result.converged for result in python_fits])
python_valid = np.concatenate([result.uncertainty_valid for result in python_fits])
print({"candidates": xfit_data.batch_size, "converged": int(python_converged.sum()),
       "valid_covariance": int(python_valid.sum()), "solver_policy": "v0.2"})
model = GaussianDipoleModel(xfit_data.images.shape[-2:], dtype=np.float64)
fit_model = np.asarray(model.evaluate(python_fits[0].parameters, mode="difference"))
_ = image_grid([xfit_data.images[0], fit_model[0]],
               ["Difference stamp", "Gaussian dipole model"],
               columns=2, diverging=True, shared_scale=True)
fit_residual = -python_fits[0].residuals[0]  # xFit returns model minus data.
stamp_peak = float(np.max(np.abs(xfit_data.images[0])))
residual_peak = float(np.max(np.abs(fit_residual)))
print({"residual_peak_image_units": residual_peak,
       "residual_peak_over_stamp_peak": residual_peak / max(stamp_peak, 1e-30)})
_ = image_grid([fit_residual],
               [f"Data − model; own scale; peak/stamp = {residual_peak / max(stamp_peak, 1e-30):.2%}"],
               columns=1, diverging=True)

### Recover the known motion, then compare executors

The stamp and model share a color range. The residual has its own labeled range
so a good fit's remaining structure stays visible; saturated residual colors do
not mean errors as large as the source.

The injected old and new centers supply an independent recovery check. xFit
positions are relative to the center of each odd-sized stamp. Add the candidate's
full-image `(x, y)` before comparing the positive lobe to the new position and the
negative lobe to the old position. These errors are in **pixels**. We report every
injected mover, including failed fits; backend agreement alone would not show
whether either calculation recovered the signal. Circular lobes can leave
orientation poorly constrained, so inspect center error separately from the
full covariance's validity flag.


In [ ]:
truth_by_id = {candidate["candidate_id"]: candidate
               for item in data_manifest["pairs"] for candidate in item["candidates"]}
recovery_rows = []
for index, candidate_id in enumerate(xfit_data.candidate_id):
    candidate = truth_by_id[str(candidate_id)]
    if candidate["kind"] != "moving_source":
        continue
    origin = np.array([candidate["x"], candidate["y"]], dtype=float)
    fitted_new = python_parameters[index, 4:6] + origin
    fitted_old = python_parameters[index, 6:8] + origin
    true_new = np.array(candidate["truth"]["target_xy"])
    true_old = np.array(candidate["truth"]["reference_xy"])
    recovery_rows.append({
        "candidate_id": str(candidate_id),
        "positive_center_error_px": float(np.linalg.norm(fitted_new - true_new)),
        "negative_center_error_px": float(np.linalg.norm(fitted_old - true_old)),
        "motion_vector_error_px": float(np.linalg.norm((fitted_new - fitted_old) - (true_new - true_old))),
        "converged": bool(python_converged[index]),
        "uncertainty_valid": bool(python_valid[index]),
    })
recovery = pd.DataFrame(recovery_rows)
display(recovery.head(12))
error_columns = ["positive_center_error_px", "negative_center_error_px", "motion_vector_error_px"]
display(recovery[error_columns].describe(percentiles=[0.5, 0.9]))
# A small histogram table includes failed fits rather than filtering them away.
bins = [0, 0.1, 0.25, 0.5, 1, 2, np.inf]
histogram = {"error_px": ["0–0.1", "0.1–0.25", "0.25–0.5", "0.5–1", "1–2", "2+"]}
for column in error_columns:
    histogram[column] = np.histogram(recovery[column], bins=bins)[0]
display(pd.DataFrame(histogram))
print({"moving_sources": len(recovery), "converged": int(recovery.converged.sum()),
       "valid_covariance": int(recovery.uncertainty_valid.sum()),
       "nonfinite_error_rows": int((~np.isfinite(recovery[error_columns])).any(axis=1).sum())})
recovery.to_csv(RUNS / "injection-recovery.csv", index=False)


In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xfit-dragon.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xfit-dragon.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp cuphoton xfit fit-dipoles \
    --executor dragon --input "$XFIT_INPUT" --model gaussian --mode difference \
    --backend cupy --compute-dtype float64 --solver-version v0.2 \
    --chunk-size 32 --max-evaluations 400 \
    --output-dir "$RUNS/xfit-dragon" --max-workers "$WORKERS" >"$RUNS/xfit-dragon.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp cuphoton xfit fit-dipoles \
    --executor dragon --input "$XFIT_INPUT" --model gaussian --mode difference \
    --backend cupy --compute-dtype float64 --solver-version v0.2 \
    --chunk-size 32 --max-evaluations 400 \
    --output-dir "$RUNS/xfit-dragon" --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp cuphoton xfit fit-dipoles \
    --executor dragon --input "$XFIT_INPUT" --model gaussian --mode difference \
    --backend cupy --compute-dtype float64 --solver-version v0.2 \
    --chunk-size 32 --max-evaluations 400 \
    --output-dir "$RUNS/xfit-dragon" --max-workers "$WORKERS"
```

In [ ]:
print("xfit-dragon: launcher-to-exit seconds:", float((RUNS / "xfit-dragon.elapsed").read_text()))
fit_table = pd.read_parquet(RUNS / "xfit-dragon" / "scientific" / "fits.parquet")
assert fit_table["candidate_id"].tolist() == xfit_data.candidate_id.tolist()
print(compare_arrays(python_parameters,
                     fit_table[list(python_fits[0].parameter_names)].to_numpy(),
                     label="xFit Python / Dragon parameters"))
np.testing.assert_array_equal(python_converged, fit_table["converged"].to_numpy())
display(fit_table[["candidate_id", "converged", "uncertainty_valid"]].head(8))
_ = summarize_run(RUNS / "xfit-dragon")

## 5. Score the candidate batch · 19–23 minutes

The small triplet-plus-xFit model was trained in preparation, with disjoint
training, validation and held-out inputs. The table below reports that particular
injection experiment. Training runs 20 GPU epochs and retains the checkpoint
with the best validation ROC AUC; the qualified run retained **epoch 1**. The
history below shows the later overfitting. Scores are uncalibrated and narrowly
clustered near 0.5; inspect their ranking and logits as well as the validation
threshold. They are not probabilities that an arbitrary sky source is real.

The Python and CLI treatments load the same checkpoint and feature schema, and
preserve the same inference minibatches. The stock dataset inference API is used
here so both treatments use the same host sigmoid and sample ordering.

In [ ]:
from cuphoton.xscan.dataset import StampDataset
from cuphoton.xscan.training import load_model_from_checkpoint, predict_dataset

training_summary = load_json(Path(prepared["model_dir"]) / "summary.json")
training_history = pd.DataFrame(load_json(Path(prepared["model_dir"]) / "history.json"))
best_epoch = int(training_summary["early_stopping"]["best_epoch"])
print({"epochs_completed": training_summary["epochs_completed"],
       "retained_best_epoch": best_epoch, "selection_metric": training_summary["best_metric"]})
# The beginning, best epoch, and end show the training trend without a long table.
history_rows = training_history["epoch"].isin([1, 2, 3, best_epoch, 10, 20])
display(training_history.loc[history_rows, ["epoch", "train_loss", "val_loss", "val_roc_auc"]])

metrics = prepared["metrics"]
if isinstance(metrics, str):
    metrics = load_json(metrics)
metric_columns = ["sample_count", "balanced_accuracy", "roc_auc", "precision", "recall", "fp", "fn"]
display(pd.DataFrame(metrics["splits"]).T.loc[["val", "test"], metric_columns])
print("Threshold selected on validation:", metrics["decision_threshold"])
BATCH = int(prepared.get("config", {}).get("batch_size", 32))
SPLIT = prepared.get("config", {}).get("inference_split", "all")
assert BATCH == 32 and SPLIT == "test"
classifier, checkpoint, performance = load_model_from_checkpoint(
    Path(prepared["model_dir"]), device=torch.device("cuda:0")
)
dataset = StampDataset(
    Path(prepared["dataset_dir"]), input_mode="triplet", split=SPLIT,
    xfit_feature_dir=Path(prepared["xfit_feature_dir"]),
    xfit_feature_names=tuple(classifier.xfit_feature_names),
)
python_predictions = predict_dataset(
    model=classifier, dataset=dataset, batch_size=BATCH,
    device=torch.device("cuda:0"), performance=performance,
)
# Fixed first two examples of each injected class, independent of model score.
prediction_rows = python_predictions["metadata_rows"]
contact_positions = [
    position
    for kind in ("moving_source", "detector_artifact", "unchanged_control")
    for position in [i for i, row in enumerate(prediction_rows) if row["kind"] == kind][:2]
]
contact_samples = [int(dataset.indices[position]) for position in contact_positions]
_ = image_grid([dataset.difference[sample] for sample in contact_samples],
               [prediction_rows[position]["kind"] + chr(10) + prediction_rows[position]["candidate_id"]
                for position in contact_positions], diverging=True)
# Rank 1 is the highest score across the entire demonstration candidate batch.
score_ranks = pd.Series(python_predictions["probabilities"]).rank(method="min", ascending=False)
print({"uncalibrated_score_min": float(np.min(python_predictions["probabilities"])),
       "uncalibrated_score_max": float(np.max(python_predictions["probabilities"]))})
# Truth is known from injection; scores, ranks and decisions come from the model.
threshold = float(metrics["decision_threshold"])
display(pd.DataFrame([
    {"candidate_id": prediction_rows[position]["candidate_id"],
     "injected_kind": prediction_rows[position]["kind"],
     "injected_label": int(python_predictions["labels"][position]),
     "model_logit": float(python_predictions["logits"][position]),
     "score_rank": int(score_ranks.iloc[position]),
     "uncalibrated_score": float(python_predictions["probabilities"][position]),
     "model_decision": bool(python_predictions["probabilities"][position] >= threshold)}
    for position in contact_positions
]))

In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xscan-dragon.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xscan-dragon.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp cuphoton xscan infer-real-bogus \
    --executor dragon --run-dir "$MODEL_DIR" --dataset-dir "$DEMO_DATASET" \
    --split test --batch-size 32 --num-workers 0 --task-batches 1 \
    --use-xfit-features --xfit-feature-dir "$XFIT_FEATURES" \
    --output-dir "$RUNS/xscan-dragon" --max-workers "$WORKERS" >"$RUNS/xscan-dragon.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp cuphoton xscan infer-real-bogus \
    --executor dragon --run-dir "$MODEL_DIR" --dataset-dir "$DEMO_DATASET" \
    --split test --batch-size 32 --num-workers 0 --task-batches 1 \
    --use-xfit-features --xfit-feature-dir "$XFIT_FEATURES" \
    --output-dir "$RUNS/xscan-dragon" --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp cuphoton xscan infer-real-bogus \
    --executor dragon --run-dir "$MODEL_DIR" --dataset-dir "$DEMO_DATASET" \
    --split test --batch-size 32 --num-workers 0 --task-batches 1 \
    --use-xfit-features --xfit-feature-dir "$XFIT_FEATURES" \
    --output-dir "$RUNS/xscan-dragon" --max-workers "$WORKERS"
```

In [ ]:
print("xscan-dragon: launcher-to-exit seconds:", float((RUNS / "xscan-dragon.elapsed").read_text()))
scientific = RUNS / "xscan-dragon" / "scientific"
np.testing.assert_array_equal(dataset.indices, np.load(scientific / "sample_index.npy"))
print(compare_arrays(python_predictions["logits"], np.load(scientific / "logits.npy"),
                     label="xScan Python / Dragon logits"))
print(compare_arrays(python_predictions["probabilities"],
                     np.load(scientific / "probabilities.npy"),
                     label="xScan Python / Dragon probabilities"))
_ = summarize_run(RUNS / "xscan-dragon")

## 6. Keep the whole pipeline in one worker · 23–31 minutes

The separate commands above make stage boundaries inspectable. The resident
pipeline processes an entire pair through xPois → xFit → feature conversion →
xScan in one worker, reusing the model and CUDA context. CuPy and Torch share
compatible device buffers via DLPack. It writes compact results instead of full
intermediate images.

This pipeline takes already aligned pairs and supplied candidate coordinates.
It does not perform xRep or detection, and its xFit feature contract is unmasked,
unweighted Gaussian fitting. Its xPois variance and mask do not become xFit
weights. The optional survey chain below has a different, richer contract.

Preparation fits each full corpus in one local xFit call. The live xFit comparison
uses 32-candidate chunks, and the resident pipeline refits each 32-candidate pair.
Poorly constrained artifact fits can vary with batch size. Standalone inference
uses the saved feature bundle, so its scores are a separate comparison from
resident feature recomputation.


In [ ]:
from cuphoton.xscan.device_pipeline import (
    DeviceWorkerContext, decode_device_pipeline_evidence, run_device_pipeline_item,
)
from cuphoton.xscan.pipeline_executor import load_pipeline_manifest

pipeline_config, pipeline_items = load_pipeline_manifest(pipeline_path)
context = DeviceWorkerContext.initialize(pipeline_config)
try:
    # Readable API example: one pair. Full-corpus timings use the CLI below.
    started = time.perf_counter()
    api_result = run_device_pipeline_item(pipeline_items[0], context)
    api_item_seconds = time.perf_counter() - started
finally:
    context.producer_stream.synchronize()
    context.consumer_stream.synchronize()
print({"context_setup_sec": context.load_seconds, "one_item_sec": api_item_seconds})
display(pd.json_normalize(api_result.transfers.to_payload()))
# Release the local model before launching fresh workers on the same GPU.
del context, classifier
cp.get_default_memory_pool().free_all_blocks()
torch.cuda.empty_cache()
gc.collect()


### Run the full corpus locally, then with Dragon

The local command is `cuphoton xscan run-pipeline --executor processes`. It uses
one persistent process on one explicitly visible GPU. Both CLI treatments run
all prepared pairs with one warmup and three measured rounds; the earlier Python
cell runs one representative pair solely to expose the API and data flow.

The local child uses the first GPU already visible to the notebook, preserving
an inherited scheduler mask. Dragon can use the configured worker count across
its selected hosts. This compares execution arrangements for the same corpus;
keep worker count and device differences visible alongside timing results.


In [ ]:
%%bash
set -euxo pipefail
# Use the notebook's first visible GPU, preserving an inherited scheduler mask.
export CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES:-0}"
export CUDA_VISIBLE_DEVICES="${CUDA_VISIBLE_DEVICES%%,*}"
trap 'tail -n 40 "$RUNS/pipeline-local.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/pipeline-local.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  cuphoton xscan run-pipeline \
    --executor processes --workers-per-gpu 1 --manifest "$PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-local --warmup-rounds 1 --measure-rounds 3 >"$RUNS/pipeline-local.log" 2>&1

In [ ]:
print("pipeline-local: launcher-to-exit seconds:", float((RUNS / "pipeline-local.elapsed").read_text()))
_ = summarize_run(RUNS / "pipeline-local")
local_rounds = sorted((RUNS / "pipeline-local" / "rounds").glob("measure-*"))
assert len(local_rounds) == 3
local_by_id = {
    item.item_id: load_json(local_rounds[-1] / "items" / item.item_id / "summary.json")
    for item in pipeline_items
}
api_arrays = decode_device_pipeline_evidence(api_result.scientific_evidence, config=pipeline_config)
cli_arrays = decode_device_pipeline_evidence(
    local_by_id[api_result.item_id]["scientific_evidence"], config=pipeline_config,
)
for name in api_arrays:
    compare_arrays(api_arrays[name], cli_arrays[name], rtol=1e-8, atol=1e-10,
                   label=f"Direct API / local CLI: {name}")

print(f"Direct API / local CLI: {len(api_arrays)} scientific arrays passed")

In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/pipeline-dragon.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/pipeline-dragon.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp cuphoton xscan run-pipeline \
    --executor dragon --manifest "$PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-dragon \
    --warmup-rounds 1 --measure-rounds 3 --max-workers "$WORKERS" >"$RUNS/pipeline-dragon.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp cuphoton xscan run-pipeline \
    --executor dragon --manifest "$PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-dragon \
    --warmup-rounds 1 --measure-rounds 3 --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp cuphoton xscan run-pipeline \
    --executor dragon --manifest "$PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-dragon \
    --warmup-rounds 1 --measure-rounds 3 --max-workers "$WORKERS"
```

In [ ]:
print("pipeline-dragon: launcher-to-exit seconds:", float((RUNS / "pipeline-dragon.elapsed").read_text()))
_ = summarize_run(RUNS / "pipeline-dragon")

### Verify the science and explain the timers

Compare the compact scientific evidence for every item, rather than accepting
only a successful launcher exit. The root execution summary also verifies worker
placement, output records and shutdown. Small images can spend more time in
launch and I/O than in arithmetic; the notebook does not promise a speedup.

The one-item Python illustration excludes process startup and result-file
publication. Use the full-corpus CLI runs for timing comparisons. Their batch
timers include dispatch, input reads, computation and worker outputs;
validation/merging follows. Show those boundaries and actual worker counts
alongside launcher-to-exit time. Do not divide unmatched timers to claim
acceleration.

GNU `time` writes each complete command's elapsed seconds to `<stage>.elapsed`;
stdout/stderr go to `<stage>.log`. The Python cell reads that timing separately
from the worker and batch timers reported by cuPhoton. The final local round is
the baseline; skip its eight self-comparisons, leaving 40 item-round comparisons.

In [ ]:
parity = []
for treatment in ("pipeline-local", "pipeline-dragon"):
    round_dirs = sorted((RUNS / treatment / "rounds").glob("measure-*"))
    assert len(round_dirs) == 3
    for round_dir in round_dirs:
        if treatment == "pipeline-local" and round_dir == local_rounds[-1]:
            continue  # This is the reference round, not an independent comparison.
        for item in pipeline_items:
            observed_payload = load_json(round_dir / "items" / item.item_id / "summary.json")
            local = local_by_id[item.item_id]
            assert observed_payload["item_id"] == local["item_id"]
            expected = decode_device_pipeline_evidence(local["scientific_evidence"],
                                                        config=pipeline_config)
            observed = decode_device_pipeline_evidence(observed_payload["scientific_evidence"],
                                                        config=pipeline_config)
            for name in expected:
                compare_arrays(expected[name], observed[name], rtol=1e-8, atol=1e-10,
                               label=f"{item.item_id}: {name}")
            parity.append({"treatment": treatment, "round": round_dir.name,
                           "item": item.item_id, "arrays_checked": len(expected), "parity": "passed"})
display(pd.DataFrame(parity))
(RUNS / "pipeline-parity.json").write_text(json.dumps(parity, indent=2) + chr(10))

### Saved features versus features recomputed by the resident pipeline

This is a diagnostic across the two feature-generation batch shapes described
above, separate from the executor-parity checks. Report score differences and
decision flips without requiring the two feature bundles to be identical.

In [ ]:
resident_scores = {}
for item in pipeline_items:
    arrays = decode_device_pipeline_evidence(
        local_by_id[item.item_id]["scientific_evidence"], config=pipeline_config,
    )
    for candidate, score in zip(item.candidates, arrays["xscan.probabilities"], strict=True):
        resident_scores[str(candidate.candidate_id)] = float(score)
score_comparison = pd.DataFrame({
    "candidate_id": [row["candidate_id"] for row in prediction_rows],
    "kind": [row["kind"] for row in prediction_rows],
    "saved_feature_score": python_predictions["probabilities"],
    "resident_score": [resident_scores[row["candidate_id"]] for row in prediction_rows],
})
score_comparison["absolute_delta"] = abs(score_comparison.saved_feature_score - score_comparison.resident_score)
score_comparison["decision_flip"] = ((score_comparison.saved_feature_score >= threshold)
                                     != (score_comparison.resident_score >= threshold))
display(score_comparison.groupby("kind").agg(
    candidates=("candidate_id", "size"), maximum_score_delta=("absolute_delta", "max"),
    decision_flips=("decision_flip", "sum"),
))
score_comparison.to_csv(RUNS / "saved-vs-resident-scores.csv", index=False)

## 7. Select native CUDA C++ implementations · 31–35 minutes

The numerical backend and the process executor are independent choices. Native
implementations use precompiled CUDA kernels and CUDA libraries while retaining
CuPy-owned buffers. `auto` does not select these paths automatically.

| Operation | Explicit selection | Boundary |
|---|---|---|
| xDR compressed-pixel restoration | `postprocess="native"` | GZIP_1/GZIP_2 tiles; reader selection is separate |
| xRep mapping expansion and reprojection | `backend="native"` | Coarse WCS evaluation remains on CPU |
| xPois source selection and constant-kernel solve | `backend="native"` | Local CLI/device API; batch CLI currently accepts CuPy, Numba-CUDA or cuTile |
| xFit Gaussian preparation, LM, and covariance | `backend="native"` | Analytic Gaussian models; also works inside Dragon workers |

The existing resident pipeline can select **native xFit**; its xPois stage remains
CuPy. No claim is made that the entire pipeline, plotting, or all dependencies
avoid JIT compilation. Compare the same model, precision, solver policy and data.


In [ ]:
compressed_target = Path(data_manifest["pairs"][0]["target_compressed"])
gzip_cupy = read_fits_images(compressed_target, [HDU], reader="xdr", device=True,
                              xdr_options={"postprocess": "fused"})
gzip_native = read_fits_images(compressed_target, [HDU], reader="xdr", device=True,
                                xdr_options={"postprocess": "native"})
print(compare_arrays(cp.asnumpy(gzip_cupy.arrays[0]), cp.asnumpy(gzip_native.arrays[0]),
                     rtol=0, atol=0, label="xDR lossless native restoration"))
print(compare_arrays(target, cp.asnumpy(gzip_native.arrays[0]),
                     rtol=0, atol=0, label="xDR native decoded / original uncompressed pixels"))

os.environ["COMPRESSED_TARGET"] = str(compressed_target)

The xDR CLI counterpart is a decoding benchmark; it does not export pixel
arrays. The API cell above checks exact decoded values. `--skip-gds-read` skips
only the separate raw-storage phase, not FITS decoding.

In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xdr-native.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xdr-native.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  cuphoton xdr benchmark-fits "$COMPRESSED_TARGET" --hdu-indices 1 \
    --iterations 1 --native-plan-threads 1 --native-read-threads 1 \
    --xdr-postprocess native --skip-gds-read --output-json "$RUNS/xdr-native.json" >"$RUNS/xdr-native.log" 2>&1

In [ ]:
native_aligned = reproject_stack(sources, stack_spec, backend="native")
print(compare_arrays(aligned.images, native_aligned.images, label="xRep CuPy / native"))


In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xrep-native.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xrep-native.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  cuphoton xrep reproject-stack \
    --inputs "$REFERENCE,$RAW_TARGET" --hdu 1 \
    --target-wcs "$REFERENCE" --target-hdu 1 \
    --backend native --fits-reader xdr --interpolation lanczos3 \
    --mapping-grid-step 1 --write-fits \
    --output-dir "$RUNS" --name xrep-native >"$RUNS/xrep-native.log" 2>&1

In [ ]:
print("xrep-native: launcher-to-exit seconds:", float((RUNS / "xrep-native.elapsed").read_text()))
native_cli_aligned = np.load(RUNS / "xrep-native" / "artifacts" / "reprojected_stack.npy")
print(compare_arrays(native_aligned.images, native_cli_aligned,
                     label="xRep native Python / CLI"))

In [ ]:
native_subtraction = solve_constant_kernel_device(
    reference_device, target_device, components, backend="native", **solve_options
)
print(compare_arrays(difference, cp.asnumpy(native_subtraction.residual),
                     label="xPois CuPy / native"))
_ = image_grid([difference, cp.asnumpy(native_subtraction.residual),
            difference - cp.asnumpy(native_subtraction.residual)],
           ["CuPy difference", "Native difference", "Numerical difference"], diverging=True)


In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xpois-native-cli.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xpois-native-cli.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  cuphoton xpois fit-kernel --backend native --solver constant \
    --reference "$REFERENCE" --reference-hdu 1 --target "$TARGET" --target-hdu 1 \
    --variance "$VARIANCE" --variance-hdu 1 --fit-mask "$FIT_MASK" \
    --kernel-height 15 --kernel-width 15 --basis-sigmas 1.5 --basis-degrees 0 \
    --background-degree 0 --fits-reader xdr --no-review \
    --output-dir "$RUNS" --name xpois-native-cli >"$RUNS/xpois-native-cli.log" 2>&1

In [ ]:
print("xpois-native-cli: launcher-to-exit seconds:", float((RUNS / "xpois-native-cli.elapsed").read_text()))
native_cli_difference = np.load(RUNS / "xpois-native-cli" / "artifacts" / "residual.npy")
print(compare_arrays(cp.asnumpy(native_subtraction.residual), native_cli_difference,
                     label="xPois native Python / CLI"))

Native xPois also moves compact-source filtering and scoring into precompiled
CUDA. Compare its selected centers and mask with the ordinary GPU-array route.
This diagnostic mask is independent of the fixed injection-exclusion mask used
for all subtraction comparisons above; we do not change the scientific inputs.
This source-selection call has no exact standalone CLI equivalent; the
`fit-kernel` command above compares the same fixed-mask solve as the Python API.


In [ ]:
from cuphoton.xpois import build_compact_source_stamp_mask

selection_options = dict(valid_mask=solve_options["fit_mask"], stamp_size=31, max_stamps=8)
# CuPy input selects the GPU-array route; this API names its modes auto/native.
selection_gpu = build_compact_source_stamp_mask(reference_device, **selection_options)
selection_native = build_compact_source_stamp_mask(
    reference_device, backend="native", **selection_options
)
assert selection_native.centers == selection_gpu.centers
np.testing.assert_array_equal(cp.asnumpy(selection_native.mask), cp.asnumpy(selection_gpu.mask))
print("Identical selected centers:", selection_native.centers)
_ = image_grid([reference, cp.asnumpy(selection_native.mask)],
               ["Reference: compact-source selection", "Native selected fit regions"], columns=2)


In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/xfit-native.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/xfit-native.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp cuphoton xfit fit-dipoles \
    --executor dragon --input "$XFIT_INPUT" --model gaussian --mode difference \
    --backend native --compute-dtype float64 --solver-version v0.2 \
    --chunk-size 32 --max-evaluations 400 \
    --output-dir "$RUNS/xfit-native" --max-workers "$WORKERS" >"$RUNS/xfit-native.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp cuphoton xfit fit-dipoles \
    --executor dragon --input "$XFIT_INPUT" --model gaussian --mode difference \
    --backend native --compute-dtype float64 --solver-version v0.2 \
    --chunk-size 32 --max-evaluations 400 \
    --output-dir "$RUNS/xfit-native" --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp cuphoton xfit fit-dipoles \
    --executor dragon --input "$XFIT_INPUT" --model gaussian --mode difference \
    --backend native --compute-dtype float64 --solver-version v0.2 \
    --chunk-size 32 --max-evaluations 400 \
    --output-dir "$RUNS/xfit-native" --max-workers "$WORKERS"
```

In [ ]:
print("xfit-native: launcher-to-exit seconds:", float((RUNS / "xfit-native.elapsed").read_text()))
native_fits = pd.read_parquet(RUNS / "xfit-native" / "scientific" / "fits.parquet")
assert native_fits["candidate_id"].tolist() == fit_table["candidate_id"].tolist()
candidate_kind = {row["candidate_id"]: row["kind"]
                  for item in data_manifest["pairs"] for row in item["candidates"]}
fit_comparison = pd.DataFrame({
    "candidate_id": fit_table["candidate_id"],
    "kind": [candidate_kind[c] for c in fit_table["candidate_id"]],
    "cupy_converged": fit_table["converged"],
    "native_converged": native_fits["converged"],
    "cupy_uncertainty_valid": fit_table["uncertainty_valid"],
    "native_uncertainty_valid": native_fits["uncertainty_valid"],
})
centers = ["x_pos", "y_pos", "x_neg", "y_neg"]
cupy_centers = fit_table[centers].to_numpy()
native_centers = native_fits[centers].to_numpy()
fit_comparison["center_max_abs_delta_px"] = np.max(np.abs(cupy_centers - native_centers), axis=1)
fit_comparison["strict_center_failure"] = ~np.isclose(
    cupy_centers, native_centers, rtol=1e-5, atol=1e-7, equal_nan=True,
).all(axis=1)
center_errors = [f"{name}_standard_error" for name in centers]
fit_comparison["cupy_center_standard_error_max_px"] = fit_table[center_errors].max(axis=1)
fit_comparison["native_center_standard_error_max_px"] = native_fits[center_errors].max(axis=1)
print("All-candidate center comparison: rtol=1e-5, atol=1e-7 pixel")
display(fit_comparison.groupby("kind").agg(
    candidates=("candidate_id", "size"),
    cupy_converged=("cupy_converged", "sum"), native_converged=("native_converged", "sum"),
    cupy_valid=("cupy_uncertainty_valid", "sum"), native_valid=("native_uncertainty_valid", "sum"),
    center_max_abs_delta_px=("center_max_abs_delta_px", "max"),
    strict_center_failures=("strict_center_failure", "sum"),
))
display(fit_comparison.nlargest(6, "center_max_abs_delta_px")[[
    "candidate_id", "kind", "center_max_abs_delta_px", "strict_center_failure",
    "cupy_center_standard_error_max_px", "native_center_standard_error_max_px",
]])
identified = ((fit_comparison["kind"] == "moving_source")
              & fit_comparison["cupy_converged"] & fit_comparison["native_converged"]
              & fit_comparison["cupy_uncertainty_valid"] & fit_comparison["native_uncertainty_valid"])
assert identified.any(), "No identified moving-source fits to compare"
print(compare_arrays(fit_table.loc[identified, centers].to_numpy(),
                     native_fits.loc[identified, centers].to_numpy(),
                     rtol=1e-4, atol=1e-5, label="xFit CuPy / native identified centers"))
fit_comparison.to_json(RUNS / "native-fit-comparison.json", orient="records", indent=2)
_ = summarize_run(RUNS / "xfit-native")

The all-candidate table reports center differences and strict comparison failure counts
for every injected class. A Gaussian dipole is a mismatched model for sharp or
track-like detector artifacts; poorly constrained fits can have large formal
center errors even when their convergence and uncertainty-valid flags are true.
Inspect those rows separately from the identified moving-source assertion.
Numerical agreement does not make a mismatched artifact fit scientifically identifiable.

To place native xFit inside the resident pipeline, only its backend field changes.
The replay below retains
solver policy v0.2 and the exact checkpoint, pair inputs, and other settings.


In [ ]:
%%bash
set -euxo pipefail
trap 'tail -n 40 "$RUNS/pipeline-native.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/pipeline-native.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp cuphoton xscan run-pipeline \
    --executor dragon --manifest "$NATIVE_PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-native \
    --warmup-rounds 1 --measure-rounds 3 --max-workers "$WORKERS" >"$RUNS/pipeline-native.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp cuphoton xscan run-pipeline \
    --executor dragon --manifest "$NATIVE_PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-native \
    --warmup-rounds 1 --measure-rounds 3 --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp cuphoton xscan run-pipeline \
    --executor dragon --manifest "$NATIVE_PIPELINE_MANIFEST" \
    --output-dir "$RUNS" --name pipeline-native \
    --warmup-rounds 1 --measure-rounds 3 --max-workers "$WORKERS"
```

In [ ]:
print("pipeline-native: launcher-to-exit seconds:", float((RUNS / "pipeline-native.elapsed").read_text()))
native_pipeline_path = Path(prepared["native_pipeline_manifest"])
_ = summarize_run(RUNS / "pipeline-native")
native_config, _ = load_pipeline_manifest(native_pipeline_path)
native_round = sorted((RUNS / "pipeline-native" / "rounds").glob("measure-*"))[-1]
native_scores = []
for item in pipeline_items:
    payload = load_json(native_round / "items" / item.item_id / "summary.json")
    observed = decode_device_pipeline_evidence(payload["scientific_evidence"], config=native_config)
    expected = decode_device_pipeline_evidence(local_by_id[item.item_id]["scientific_evidence"],
                                                config=pipeline_config)
    native_scores.append(compare_arrays(
        expected["xscan.probabilities"], observed["xscan.probabilities"],
        rtol=1e-4, atol=1e-5, label=f"{item.item_id}: CuPy/native pipeline scores"
    ))
display(pd.DataFrame(native_scores))

## Optional: discover candidates and build a richer catalog · 5–7 minutes

The existing [transient-chain example](../transient-chain/README.md) starts from
FITS pairs, aligns them, constructs a fit mask, detects residual peaks,
and fits candidates with mask/variance handling, quality flags, propagated
uncertainties, and single-source refits. It also supports a compatible image-only
checkpoint. Our injection-trained **fusion** checkpoint is a different model
contract, so this appendix intentionally ends after xFit. The example also offers
star-based registration refinement; the prepared injection demo disables it so
fitting an affine transform cannot absorb the deliberately injected motion.

The chain is an example application with its own `launch.py`, not an invented
`cuphoton` subcommand. Its present numerical path is CuPy; it does not expose the
native selectors demonstrated above. The uncertainty budget describes the
supplied noise model, and correlated or reused background pixels still limit
interpretation. The inputs here remain controlled injections on real backgrounds.

Set `CUPHOTON_DEMO_TRANSIENT_CHAIN=1` before starting Jupyter to execute the
optional Bash cell. The separate `examples/transient-chain` package must be
installed in the shared environment, as shown in the README.

In [ ]:
chain = prepared.get("transient_chain")
RUN_TRANSIENT_CHAIN = os.environ.get("CUPHOTON_DEMO_TRANSIENT_CHAIN", "0") == "1"
if RUN_TRANSIENT_CHAIN and chain is None:
    raise RuntimeError("Optional chain inputs are missing; rerun prepare.py in a new output directory.")
if chain is not None:
    os.environ.update({
        "TRANSIENT_LAUNCHER": str(REPO / "examples" / "transient-chain" / "launch.py"),
        "TRANSIENT_CONFIG": chain["config"], "TRANSIENT_SELECTIONS": chain["selections"],
    })

In [ ]:
%%bash
set -euxo pipefail
if [[ ${CUPHOTON_DEMO_TRANSIENT_CHAIN:-0} != 1 ]]; then
    echo "Optional transient-chain skipped; set CUPHOTON_DEMO_TRANSIENT_CHAIN=1 to run."
    exit 0
fi
trap 'tail -n 40 "$RUNS/transient-chain.log"' EXIT
/usr/bin/time -f '%e' -o "$RUNS/transient-chain.elapsed" \
  timeout --signal=TERM --kill-after=10s 600s \
  dragon -s -t tcp -o tcp python "$TRANSIENT_LAUNCHER" \
    --executor dragon --config "$TRANSIENT_CONFIG" --selections "$TRANSIENT_SELECTIONS" \
    --output-dir "$RUNS" --name transient-chain --max-workers "$WORKERS" >"$RUNS/transient-chain.log" 2>&1

For an allocated cluster, replace the `dragon -s ...` command above with
one of these complete commands; retain the Bash cell's timing and log redirection.
Shared absolute paths must exist on every host.

```bash
# Slurm: run inside an interactive allocation.
dragon --wlm slurm --nodes "$NODES" -t tcp -o tcp python "$TRANSIENT_LAUNCHER" \
    --executor dragon --config "$TRANSIENT_CONFIG" --selections "$TRANSIENT_SELECTIONS" \
    --output-dir "$RUNS" --name transient-chain --max-workers "$WORKERS"

# SSH: use the host file for dedicated, authorized hosts.
dragon --wlm ssh --nodes "$NODES" --hostfile "$HOSTFILE" -t tcp -o tcp python "$TRANSIENT_LAUNCHER" \
    --executor dragon --config "$TRANSIENT_CONFIG" --selections "$TRANSIENT_SELECTIONS" \
    --output-dir "$RUNS" --name transient-chain --max-workers "$WORKERS"
```

In [ ]:
if RUN_TRANSIENT_CHAIN:
    print("transient-chain: launcher-to-exit seconds:", float((RUNS / "transient-chain.elapsed").read_text()))
    _ = summarize_run(RUNS / "transient-chain")
    catalogs = list((RUNS / "transient-chain").glob("**/catalog.parquet"))
    catalog = pd.read_parquet(catalogs[0])
    display(catalog[["candidate_id", "x", "y", "peak_abs_sigma", "amplitude", "amplitude_err",
                     "probability", "decision"]].head(12))

The completed run leaves source provenance, manifests, model evaluation, launch
logs, scientific comparisons, and worker records beside the data. The plots show
what each stage calculated; the recorded host/GPU identities show where it ran.
For larger observations, retain the same contracts and increase the number of
independent image pairs before changing worker sharing or transport settings.

Further reading: [Dragon architecture](../../docs/dragon.md),
[launch guide](../../docs/distributed-execution.md),
[transport and timing](../../docs/dragon-performance.md),
[native xFit](../../docs/components/xfit.md#optional-native-cuda-backend), and
[xImage reconstruction](../../docs/components/ximage.md) for the separate problem
of reconstructing a common scene from registered exposures with supplied PSFs.
